# Lab 3: Bottom-Up Steel Emissions with Uncertainty


---

## What this lab is

* **Part A: emissions pipeline.** Same tasks, same order, same variable names as Lab 3: operating plants 2020-2030, China utilisation by technology, plant-level production, emission factors, company emissions, projection to 2030.
* **Part B : uncertainty quantification.** You will *interrogate* your own Lab 3 numbers with the tools of the Session 4 lecture: where do the numbers come from, how sure can we be, and what would a risk committee need to know?


## Learning goals

1. Rebuild a bottom-up emissions pipeline from plant-level data (Lab 3).
2. Tell **aleatoric, epistemic and scenario** uncertainty apart.
3. Spot a **data-coverage problem** by checking against official statistics.
4. Give a **range instead of a single number** (bootstrap and simulation) and find which input matters most.
5. Compare **scenarios** and communicate the result with calibrated words.

## Conventions

* Questions are **[QUAL]** (2-4 sentences) or **[QUANT]** (a number from your code plus one sentence).
* Write each answer in the cell right under the question.
* `ttpa` = thousand tonnes per annum. Emissions in Part A are in tCO2; in Part B we switch to **MtCO2**.
* Boxes marked **[OPTIONAL]** can be skipped.
* Run **Kernel -> Restart & Run All** before submitting.

## Submission

Full name: **Capucine GROS**  Student ID: **B00822768**

**Group / repo name:** aidams-lab3-gros

**Submitter (one person):**  Capucine GROS

**Repo URL:**  https://github.com/CapucineGros/aidams-lab3-gros

### What to submit

- This notebook (`lab3.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).


**!!!! Note on data and results: !!!!**

We used Plant-level data – Global Iron and Steel Tracker – September 2026 (V1) (the lab refers to the 2025 version), so some column names differ (GEM plant ID, Country/area) and were renamed to match the lab. China has no crude steel production reported after 2024, so actual_years = 2020-2024. Our BF-BOF utilisation in 2020 is about 26% (the lab text mentions about 57%), and the worldsteel figure for China in 2022 is 1,018 Mt rather than 1,015 Mt. Our numbers may therefore differ slightly from the expected ones.

**Source:**

worldsteel, World Steel in Figures (2021-2023 editions); values read via
https://www.steelorbis.com/steel-production-by-country/ and
https://recyclinginternational.com/business/global-steel-production-static-in-2023/56001/

## Setup (given)

In [5]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

DATA_PATH = Path(os.environ.get("GIST_PATH", "Plant-level data - Global Iron and Steel Tracker - September 2026 (V1).xlsx"))
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH.resolve()}")

plant_data = pd.read_excel(DATA_PATH, sheet_name="Plant data")
capacity_data = pd.read_excel(DATA_PATH, sheet_name="Plant capacities and status")
production_data = pd.read_excel(DATA_PATH, sheet_name="Plant production")

years = list(range(2020, 2031))
SEED = 42
print(plant_data.shape, capacity_data.shape, production_data.shape)

(1293, 44) (1848, 16) (1393, 10)


# PART A. Emissions pipeline

## A1. Operating plants 2020-2030

**Goal.** Create `operational_plants`: one row per (plant, year) for the years in `years`, with the plant's `Country/Area`, `Status` and nominal crude steel capacity.
A plant operates in year *t* if `start year <= t <= retired year`. **Assumption (Lab 3):** a missing end date means the plant continues operating.

In [6]:
def to_year(s):
    # Convert ints, datetimes and strings such as "unknown" into a numeric year (NaN if unknown)
    dt = pd.to_datetime(s, errors="coerce")
    num = pd.to_numeric(s, errors="coerce")
    return num.where(num.notna(), dt.dt.year)

cap = capacity_data.rename(columns={"GEM plant ID": "Plant ID", "Country/area": "Country/Area",
                                    "Nominal crude steel capacity (ttpa)": "capacity"})
cap["capacity"] = pd.to_numeric(cap["capacity"], errors="coerce")
cap["start year"] = to_year(cap["Start date"])

# Retired date lives in the plant sheet
ret = plant_data.rename(columns={"GEM plant ID": "Plant ID"})[["Plant ID", "Retired date"]]
ret["retired year"] = to_year(ret["Retired date"])
cap = cap.merge(ret[["Plant ID", "retired year"]], on="Plant ID", how="left")

frames = []
for t in years:
    # Operating in t if start <= t <= retired; a missing end date means still operating
    m = (cap["start year"] <= t) & (cap["retired year"].isna() | (cap["retired year"] >= t))
    tmp = cap.loc[m, ["Plant ID", "Country/Area", "Status", "Main production equipment", "capacity"]].copy()
    tmp["Year"] = t
    frames.append(tmp)

operational_plants = pd.concat(frames, ignore_index=True)
print(operational_plants.shape)

(11498, 6)


### Q1 [QUANT] Is every "operating" plant really operating?

Using your `operational_plants` for **China**: for the years **2020 and 2026**, how many plants and how much capacity (ttpa) are counted as operating although their **`Status`** is not an operating status (look at the values of `Status` first)?
Give the share of the Chinese total.

In [7]:
op_status = ["operating", "operating pre-retirement"]
china = operational_plants[operational_plants["Country/Area"] == "China"]
for y in (2020, 2026):
    d = china[china["Year"] == y]
    bad = d[~d["Status"].isin(op_status)]
    print(y, d["Plant ID"].nunique(), d["capacity"].sum(),
          bad["Plant ID"].nunique(), bad["capacity"].sum(),
          bad["Plant ID"].nunique() / d["Plant ID"].nunique(),
          bad["capacity"].sum() / d["capacity"].sum())

2020 225 786449.0 105 176621.0 0.4666666666666667 0.22458036058282227
2026 208 767033.0 70 105301.0 0.33653846153846156 0.13728353278151006


**Your answer (Q1):**

In 2020, 105 of the 225 Chinese plants counted as operating (46.7%) have a non-operating Status, representing 176,621 ttpa out of 786,449 ttpa (22.5% of capacity). In 2026, it is 70 of 208 plants (33.7%) and 105,301 ttpa out of 767,033 ttpa (13.7%). Most of them are retired plants (100 in 2020, 64 in 2026) whose end date is missing, so the “missing end date = still operating” assumption keeps them alive. The rest are cancelled, mothballed, announced or under-construction plants. This inflates the capacity, and therefore the estimated production and emissions.

## A2. China production and capacity

**Goal.** Create
* `crude_steel`: plant-year production (crude steel only), with `Country/Area`, `Owner`, `Parent` and `Main production equipment` attached;
* `china_production`: China total production by year (a Series indexed by year);
* `china_operational`: the rows of `operational_plants` for China;
* `china_capacity`: China total capacity by year.

In [8]:
pdm = plant_data.rename(columns={"GEM plant ID": "Plant ID", "Country/area": "Country/Area",
                                 "Parent (English)": "Parent"})
prod = production_data.rename(columns={"GEM plant ID": "Plant ID"})

# Crude steel only, wide -> long for the years of interest
prod = prod[prod["Type of production"] == "Crude steel production (ttpa)"]
year_cols = [c for c in prod.columns if isinstance(c, int) and c in years]
crude_steel = prod.melt(id_vars=["Plant ID"], value_vars=year_cols,
                        var_name="Year", value_name="production_ttpa")
crude_steel["production_ttpa"] = pd.to_numeric(crude_steel["production_ttpa"], errors="coerce")
crude_steel = crude_steel.dropna(subset=["production_ttpa"])

# One row per plant-year (a few plants have several crude steel rows)
crude_steel = crude_steel.groupby(["Plant ID", "Year"], as_index=False)["production_ttpa"].sum()

# Attach country, owner, parent and equipment
crude_steel = crude_steel.merge(
    pdm[["Plant ID", "Country/Area", "Owner", "Parent", "Main production equipment"]],
    on="Plant ID", how="left")

china_production = (crude_steel[crude_steel["Country/Area"] == "China"]
                    .groupby("Year")["production_ttpa"].sum())

china_operational = operational_plants[operational_plants["Country/Area"] == "China"].copy()
china_capacity = china_operational.groupby("Year")["capacity"].sum()

print(china_production)
print(china_capacity)

Year
2020   179,824.00
2021    54,817.00
2022   122,752.00
2023    97,068.00
2024    77,204.00
Name: production_ttpa, dtype: float64
Year
2020   786,449.00
2021   789,822.00
2022   791,346.00
2023   775,558.00
2024   769,038.00
2025   767,938.00
2026   767,033.00
2027   764,633.00
2028   764,633.00
2029   764,633.00
2030   764,633.00
Name: capacity, dtype: float64


## A3. Utilisation by technology

**Goal.** Compute the utilisation rate (production / capacity) separately for **BF-BOF** and **EAF** (and a residual class "Other").
Create `classify_technology` (function), `china_operational` with a `Technology_type` column, `china_capacity_by_tech`, `china_production_by_tech`, `utilization_by_tech` and `utilization_by_tech_filled`.

In [9]:
def classify_technology(equipment):
    # Group the equipment string into BF-BOF, EAF or Other
    if pd.isna(equipment):
        return "Other"
    e = str(equipment)
    if "BF" in e or "BOF" in e:
        return "BF-BOF"
    if "EAF" in e:
        return "EAF"
    return "Other"

china_operational["Technology_type"] = china_operational["Main production equipment"].apply(classify_technology)

china_capacity_by_tech = (china_operational.groupby(["Year", "Technology_type"])["capacity"]
                          .sum().unstack(fill_value=0))

cs = crude_steel[crude_steel["Country/Area"] == "China"].copy()
cs["Technology_type"] = cs["Main production equipment"].apply(classify_technology)
china_production_by_tech = (cs.groupby(["Year", "Technology_type"])["production_ttpa"]
                            .sum().unstack(fill_value=0).reindex(china_capacity_by_tech.index))

utilization_by_tech = (china_production_by_tech / china_capacity_by_tech).replace([np.inf, -np.inf], np.nan)

# Fill years without data (2025-2030) with the last observed rate; 0 means "not reported", not "idle"
utilization_by_tech_filled = utilization_by_tech.replace(0, np.nan).ffill().bfill().fillna(0)
print(utilization_by_tech_filled)

Technology_type  BF-BOF  EAF  Other
Year                               
2020               0.26 0.02   0.00
2021               0.08 0.02   0.00
2022               0.17 0.03   0.00
2023               0.14 0.04   0.00
2024               0.11 0.03   0.00
2025               0.11 0.03   0.00
2026               0.11 0.03   0.00
2027               0.11 0.03   0.00
2028               0.11 0.03   0.00
2029               0.11 0.03   0.00
2030               0.11 0.03   0.00


## A4. Plant-level production

**Goal.** Attribute each technology's utilisation rate to its plants: add `Utilization_rate_tech` and `Estimated_production_ttpa` (= capacity x utilisation) to `china_operational`.

In [10]:
# Attach each technology's utilisation rate (by year) to its plants
u = utilization_by_tech_filled.stack().rename("Utilization_rate_tech").reset_index()
china_operational = china_operational.merge(u, on=["Year", "Technology_type"], how="left")

# Estimated production = capacity x utilisation (missing capacity = 0)
china_operational["Estimated_production_ttpa"] = (
    china_operational["capacity"].fillna(0) * china_operational["Utilization_rate_tech"]
)

print(china_operational.groupby("Year")["Estimated_production_ttpa"].sum())

Year
2020   181,406.25
2021    54,817.00
2022   122,752.00
2023    97,068.00
2024    77,204.00
2025    77,081.83
2026    76,868.43
2027    76,601.87
2028    76,601.87
2029    76,601.87
2030    76,601.87
Name: Estimated_production_ttpa, dtype: float64


## A5. Emission factors and plant emissions

**Goal.** Create `emission_factors` (dict), `map_emission_factor` (function), and add `Emission_factor_tCO2_per_t` and `Estimated_emissions_tCO2` to `china_operational`.

Use these emissions factors (tCO2 per tonne crude steel, after Hasanbeigi 2019), keyed by the **exact** equipment string, with **1.6** for anything else:

| `BF; BOF` 2.1 | `BF; BOF; EAF` 1.9 | `BF; BOF; EAF; DRI` 1.9 | `BF; BOF; DRI` 2.0 | `BF; EAF` 1.9 |
|---|---|---|---|---|
| `BF` 2.2 | `EAF` 0.4 | `EAF; DRI` 1.2 | `DRI` 1.3 | `Steel other/unspecified` 1.6 |

> **Hint.** Emissions (t) = production (ttpa) x 1,000 x factor. `Series.map` with a function, or `dict.get(key, 1.6)`.

In [11]:
emission_factors = {
    "BF; BOF": 2.1, "BF; BOF; EAF": 1.9, "BF; BOF; EAF; DRI": 1.9, "BF; BOF; DRI": 2.0,
    "BF; EAF": 1.9, "BF": 2.2, "EAF": 0.4, "EAF; DRI": 1.2, "DRI": 1.3,
    "Steel other/unspecified": 1.6,
}

def map_emission_factor(equipment):
    # Exact string match, 1.6 for anything else
    return emission_factors.get(equipment, 1.6)

china_operational["Emission_factor_tCO2_per_t"] = china_operational["Main production equipment"].map(map_emission_factor)

# tCO2 = production (ttpa) x 1,000 x factor
china_operational["Estimated_emissions_tCO2"] = (
    china_operational["Estimated_production_ttpa"] * 1000 * china_operational["Emission_factor_tCO2_per_t"]
)

print((china_operational.groupby("Year")["Estimated_emissions_tCO2"].sum() / 1e6).round(1))

Year
2020   363.10
2021   107.70
2022   242.10
2023   189.30
2024   151.30
2025   151.10
2026   150.50
2027   150.00
2028   150.00
2029   150.00
2030   150.00
Name: Estimated_emissions_tCO2, dtype: float64


## A6. Company emissions

**Goal.** Create `Company` (= `Parent`, or `Owner` when `Parent` is missing), `company_emissions` (columns `Year`, `Company`, `estimated_emissions_tCO2`) and `top_companies_latest` (top 10 in the **latest year**).

In [12]:
# Attach owner / parent to the plant-year rows
china_operational = china_operational.merge(pdm[["Plant ID", "Owner", "Parent"]], on="Plant ID", how="left")

# Company = Parent, or Owner when Parent is missing
china_operational["Company"] = china_operational["Parent"].fillna(china_operational["Owner"])

company_emissions = (china_operational.groupby(["Year", "Company"], as_index=False)["Estimated_emissions_tCO2"].sum()
                     .rename(columns={"Estimated_emissions_tCO2": "estimated_emissions_tCO2"}))

latest_year = company_emissions["Year"].max()  # 2030
top_companies_latest = (company_emissions[company_emissions["Year"] == latest_year]
                        .nlargest(10, "estimated_emissions_tCO2"))
print(top_companies_latest)

      Year                                            Company  \
1960  2030                   Beijing Shougang Co Ltd [100.0%]   
2067  2030              Maanshan Iron & Steel Co Ltd [100.0%]   
2059  2030                       Jingye Group Co Ltd [100.0%]   
2051  2030                  Hunan Valin Steel Co Ltd [100.0%]   
1986  2030  Fujian Metallurgical Holding Co Ltd [53.4%]; u...   
2009  2030    Guangxi Shenglong Metallurgical Co Ltd [100.0%]   
1952  2030               Baoshan Iron & Steel Co Ltd [100.0%]   
1991  2030  Gansu Jiu Steel Group Hongxing Iron and Steel ...   
1979  2030  Citic Pacific Special Steel Group Co Ltd [100.0%]   
2017  2030                         HBIS Group Co Ltd [100.0%]   

      estimated_emissions_tCO2  
1960              5,020,074.91  
2067              3,927,653.30  
2059              3,792,211.23  
2051              3,790,045.49  
1986              3,754,827.22  
2009              3,681,125.41  
1952              3,518,494.98  
1991              3,

### Q2 [QUAL] Look at the company names

Read the names in `top_companies_latest`. Give **two data-quality problems** you notice, and explain how each could bias a *financed-emissions* figure for a lender (hint: ownership shares, joint ventures).

**Your answer (Q2):**

(1) Ownership shares are embedded in the name. Names contain strings like “[100.0%]” or “Fujian Metallurgical Holding Co Ltd [53.4%]; unknown [46.6%]”. The same company can appear under several labels, and a joint venture is attributed entirely to one label. A lender financing the 53.4% shareholder would be assigned the whole plant’s emissions, or 0% of the unknown partner’s share, so its financed emissions would be over- or under-stated.

(2) Subsidiaries are not consolidated at group level. Entities such as Maanshan Iron & Steel and Baoshan Iron & Steel (listed separately) belong to larger groups, and Parent is sometimes the listed subsidiary rather than the ultimate parent. A group’s emissions are split across several rows, so exposure to that group is under-estimated, while a loan made to one subsidiary may be attributed against the wrong counterparty.

## A7. Projection to 2030

**Goal.** Create `actual_years` (years with production data), `forecast_years` (the years after the last observed one), `projected_company_emissions` (company emissions in the forecast years) and a function `top_company_emissions(df, year)` returning the top 5 companies of that year. Print the top 5 for the first three forecast years.

In [13]:
# Years with observed production for China (2020-2024), forecast = years after the last one
actual_years = sorted(china_production.index.tolist())
forecast_years = [y for y in years if y > max(actual_years)]  # 2025-2030

projected_company_emissions = company_emissions[company_emissions["Year"].isin(forecast_years)]

def top_company_emissions(df, year):
    # Top 5 companies of a given year
    return df[df["Year"] == year].nlargest(5, "estimated_emissions_tCO2")

for y in forecast_years[:3]:
    print(y)
    print(top_company_emissions(projected_company_emissions, y))

2025
      Year                                            Company  \
1014  2025                   Beijing Shougang Co Ltd [100.0%]   
1123  2025              Maanshan Iron & Steel Co Ltd [100.0%]   
1115  2025                       Jingye Group Co Ltd [100.0%]   
1106  2025                  Hunan Valin Steel Co Ltd [100.0%]   
1040  2025  Fujian Metallurgical Holding Co Ltd [53.4%]; u...   

      estimated_emissions_tCO2  
1014              5,020,074.91  
1123              3,927,653.30  
1115              3,792,211.23  
1106              3,790,045.49  
1040              3,754,827.22  
2026
      Year                                            Company  \
1207  2026                   Beijing Shougang Co Ltd [100.0%]   
1314  2026              Maanshan Iron & Steel Co Ltd [100.0%]   
1306  2026                       Jingye Group Co Ltd [100.0%]   
1298  2026                  Hunan Valin Steel Co Ltd [100.0%]   
1233  2026  Fujian Metallurgical Holding Co Ltd [53.4%]; u...   

      esti

### Q3 [QUAL] Forecast, projection or scenario?

The top-5 tables for the first projected years are (almost) identical. (a) Why? (b) Is this a **forecast**, a **projection** or a **scenario**, and what is it conditional on? (c) Name **three things** a fan chart would tell a risk manager that this table does not.

**Your answer (Q3):**

(a) The tables are almost identical because the projection only changes through the plant list (capacity) while the utilisation rate of each technology is frozen at its last observed value (2024). No plant opens or closes in those years for these companies, so their capacity, and hence emissions, stay the same.

(b) It is a scenario (a conditional projection), not a forecast: it says “if utilisation stays at the 2024 level and the plant list and emission factors stay as they are, then emissions are X”. It carries no probability and is conditional on constant utilisation, no new closures or openings beyond the dataset, and fixed emission factors.

(c) A fan chart would show a risk manager: (1) the range of plausible values around the central path, not a single number; (2) how that uncertainty widens with the horizon; (3) the probability of exceeding a given threshold (e.g. a budget or a past level), which a single table cannot give.

---
# PART B. How much can we trust these numbers?

You now have company emissions as **single numbers**. Part B asks three simple questions:

1. **Do the numbers agree with an outside reference?** (reality check)
2. **How wide is the range of plausible values?** (an interval, found by simulation)
3. **What does the range depend on?** (inputs and scenarios)

**Three words you need**

| Word | Meaning | Example |
|---|---|---|
| **Aleatoric** uncertainty | pure randomness, more data does *not* remove it | will a given year be a strong or a weak year for steel? |
| **Epistemic** uncertainty | we lack knowledge, more data or better models *can* reduce it | what is the true average emission factor of a blast furnace? |
| **Scenario** uncertainty | depends on future choices, it is not a probability | will China retire old blast furnaces by 2030? |

A **90% interval** is a range that contains the true value in 90% of cases *if our assumptions are right*.

## B0. Think first

> **Pause & think.** No code.

### Q4 [QUAL] Which kind of uncertainty?

For each input of your pipeline, write **A** (aleatoric), **E** (epistemic) or **S** (scenario), and say whether **more data could reduce it** (yes / no). One short line each.

| Input | A / E / S | More data helps? |
|---|---|---|
| Utilisation rate of a blast furnace next year | | |
| Emission factor of BF-BOF steel | | |
| Share of plants that report their production | | |
| Steel demand in 2030 | | |

**Your answer (Q4):**

| Input | A / E / S | More data helps? |
|---|---|---|
| Utilisation rate of a blast furnace next year | A | **No:** the year-to-year randomness (strong or weak year, demand shocks) remains however much history we have. |
| Emission factor of BF-BOF steel | E | **Yes:** it is a fixed but unknown quantity, and plant-level measurements or better models narrow it down. |
| Share of plants that report their production | E | **Yes:** it is a lack of knowledge (coverage), and more reporting or other sources reduce it. |
| Steel demand in 2030 | S | **No:** it depends on future choices (policy, capacity retirement, economy), so past data cannot settle it; we compare scenarios instead. |

## B1. Reality check against official statistics

Official crude steel production of China (**rounded worldsteel figures: verify them and cite your source**):

| Year | 2020 | 2021 | 2022 | 2023 |
|---|---|---|---|---|
| Mt | 1,065 | 1,035 | 1,015 | 1,020 |

**Goal.** Create `official_kt` (dict, in **ttpa**: 1 Mt = 1,000 ttpa) and a DataFrame `check` with, for 2020-2023: the production in your dataset (`china_production`), the official value, the **coverage** (dataset / official) and the **implied utilisation** (official / `china_capacity`).

> **Hint.** `pd.Series(official_kt)`, `pd.DataFrame({...})`, `.dropna()`.

In [14]:
# Official worldsteel figures for China, in ttpa (1 Mt = 1,000 ttpa)
official_kt = {2020: 1_065_000, 2021: 1_035_000, 2022: 1_018_000, 2023: 1_019_000}

check = pd.DataFrame({
    "dataset_kt": china_production,
    "official_kt": pd.Series(official_kt),
    "capacity_kt": china_capacity,
}).dropna(subset=["official_kt"])

check["coverage"] = check["dataset_kt"] / check["official_kt"]
check["implied_utilization"] = check["official_kt"] / check["capacity_kt"]
print(check)

# Bonus for Q5(b): number of Chinese plants reporting crude steel production
print(crude_steel[crude_steel["Country/Area"] == "China"].groupby("Year")["Plant ID"].nunique())

      dataset_kt  official_kt  capacity_kt  coverage  implied_utilization
2020  179,824.00 1,065,000.00   786,449.00      0.17                 1.35
2021   54,817.00 1,035,000.00   789,822.00      0.05                 1.31
2022  122,752.00 1,018,000.00   791,346.00      0.12                 1.29
2023   97,068.00 1,019,000.00   775,558.00      0.10                 1.31
Year
2020    35
2021    15
2022    21
2023    21
2024    11
Name: Plant ID, dtype: int64


### Q5 [QUANT + QUAL] What does your pipeline miss?

(a) Give the **coverage** and **implied utilisation** for 2023, and the factor by which your Part A pipeline **under-estimates** China's production. (b) The BF-BOF "utilisation" in Part A drops from about 57% to about 8% between 2020 and 2021. Is that a real change in how furnaces are run, or something else? Which kind of uncertainty is it (A, E or S)?

**Your answer (Q5):**

(a) In 2023 the dataset covers only 9.5% of official production (97,068 vs 1,019,000 ttpa), so the Part A pipeline under-estimates China’s production by a factor of about 10.5. The implied utilisation (official production / dataset capacity) is 131%, which is impossible for real plants: the tracked capacity (775,558 ttpa) is itself too low, because the dataset does not cover all Chinese steel capacity.

(b) The drop of BF-BOF “utilisation” (about 26% to 8% in my run) is not a real change in how furnaces are run. It is a data artefact: production is reported by only 35 Chinese plants in 2020 but 15 in 2021, while the capacity in the denominator keeps counting all plants. It is epistemic (E) uncertainty: a lack of knowledge about which plants report, which more or better data could reduce.

## B2. Fix the denominator: compare like with like

Use only plants that **report production** *and* have a capacity ("matched" plants), so that numerator and denominator describe the same plants.

**Goal.** Create `obs_matched` (one row per plant-year with production **and** capacity > 0, with `Technology_type` and `u` = production / capacity) and `utilization_matched` (year x technology: **sum of production / sum of capacity** of the matched plants).

> **Hint.** Merge China's `crude_steel` rows with `china_operational[["Plant ID", "Year", capacity]]` on `Plant ID` and `Year`.

In [15]:
# Capacity per plant-year (a plant can have several capacity rows)
cap_py = china_operational.groupby(["Plant ID", "Year"], as_index=False)["capacity"].sum()

cs = crude_steel[crude_steel["Country/Area"] == "China"].copy()
cs["Technology_type"] = cs["Main production equipment"].apply(classify_technology)

# Matched plants: production AND capacity > 0
obs_matched = cs.merge(cap_py, on=["Plant ID", "Year"], how="inner")
obs_matched = obs_matched[obs_matched["capacity"] > 0].copy()
obs_matched["u"] = obs_matched["production_ttpa"] / obs_matched["capacity"]

# Sum of production / sum of capacity, by year and technology
g = obs_matched.groupby(["Year", "Technology_type"])[["production_ttpa", "capacity"]].sum()
utilization_matched = (g["production_ttpa"] / g["capacity"]).unstack()
print(utilization_matched)

Technology_type  BF-BOF  EAF
Year                        
2020               0.83  NaN
2021               0.75  NaN
2022               0.92 0.69
2023               0.77 0.59
2024               0.85 0.60


### Q6 [QUANT + QUAL] Naive or matched?

(a) For **BF-BOF in 2023**, report the Part A (naive) utilisation, the matched utilisation, and the official-implied utilisation (from `check`). (b) Which would you use for projections? In one sentence, say what bias could **remain** in the matched estimate (hint: which plants report their production?).

In [16]:
naive = utilization_by_tech.loc[2023, "BF-BOF"]
matched = utilization_matched.loc[2023, "BF-BOF"]
implied = check.loc[2023, "implied_utilization"]
print(f"naive {naive:.1%} | matched {matched:.1%} | official-implied {implied:.1%}")

naive 13.8% | matched 77.0% | official-implied 131.4%


**Your answer (Q6):**

(a) For BF-BOF in 2023: the naive Part A utilisation is 13.8%, the matched utilisation is 77.0%, and the official-implied utilisation (national official production / dataset capacity, from check) is 131.4%. Note that the last one is for all of China, not only BF-BOF.

(b) I would use the matched estimate for projections, because numerator and denominator then describe the same plants, whereas the naive one divides the production of a few reporting plants by the capacity of all plants. A bias can remain: plants that report their production are probably not representative (likely larger, better-run or more transparent plants), so the matched utilisation may be too high for the non-reporting plants (selection bias).

## B3. A range from a sample: the bootstrap

We have a matched utilisation for BF-BOF, but it comes from a **sample** of plants. A different sample would give a slightly different value. How different? The **bootstrap** answers it in four steps:

1. Draw plants **with replacement** from your sample, as many as the sample contains.
2. Recompute the average utilisation (sum of production / sum of capacity).
3. Repeat **1,000 times**.
4. The 5th and 95th percentiles of the 1,000 values form a **90% interval**.

**Goal.** Write a function `boot_mean_u(df, n=1000)` that returns the `n` bootstrap values. Use it to create `ci_all` (interval for **all** matched BF-BOF plants) and `ci_small` (interval for **10 randomly chosen** BF-BOF plants).

> **Hint.** Sum production and capacity **per plant** first (`groupby("Plant ID")`), then resample plant numbers with `np.random.default_rng(42).integers(0, n_plants, size=(n, n_plants))`; the values of one draw are `prod[idx].sum(1) / cap[idx].sum(1)`. `np.percentile(values, [5, 95])`.

In [17]:
def boot_mean_u(df, n=1000):
    # Bootstrap of sum(production) / sum(capacity): resample plants with replacement
    per_plant = df.groupby("Plant ID")[["production_ttpa", "capacity"]].sum()
    prod = per_plant["production_ttpa"].to_numpy()
    cap = per_plant["capacity"].to_numpy()
    n_plants = len(per_plant)
    rng = np.random.default_rng(42)
    idx = rng.integers(0, n_plants, size=(n, n_plants))
    return prod[idx].sum(1) / cap[idx].sum(1)

bf = obs_matched[obs_matched["Technology_type"] == "BF-BOF"]

# All matched BF-BOF plants
ci_all = np.percentile(boot_mean_u(bf), [5, 95])

# 10 randomly chosen BF-BOF plants
rng = np.random.default_rng(SEED)
chosen = rng.choice(bf["Plant ID"].unique(), size=10, replace=False)
ci_small = np.percentile(boot_mean_u(bf[bf["Plant ID"].isin(chosen)]), [5, 95])

print(ci_all, ci_all[1] - ci_all[0])
print(ci_small, ci_small[1] - ci_small[0])

[0.77901244 0.86375917] 0.08474672816565898
[0.69752764 0.89858925] 0.20106161125471345


### Q7 [QUANT + QUAL] Small sample, wide interval

(a) Report `ci_all` and `ci_small` and their widths. (b) Why is the second wider? (c) In one sentence, what does "90% interval" mean here? (d) Which kind of uncertainty (A or E) shrinks when you have more plants?

**Your answer (Q7):**

(a) ci_all = [77.9%, 86.4%], width about 8.5 points. ci_small = [69.8%, 89.9%], width about 20.1 points.

(b) The second is wider because it is based on only 10 plants: each resample can include a different mix of very different plants (large or small, high or low utilisation), so the ratio moves a lot. With more plants these differences average out.

(c) A 90% interval means: if our assumptions are right (the sample is representative and the plants are drawn independently), then in 90% of cases such a range would contain the true average utilisation.

(d) Epistemic (E) uncertainty shrinks when we have more plants. The aleatoric part (year-to-year randomness) does not.

> ### [OPTIONAL] Bayesian version of the same interval
>
> A **credible interval** answers a more direct question: *given the data and what I believed before, where is the true average?* We start from a **prior guess** (mean $\mu_0$, worth $\kappa_0$ observations) and update it with the data (mean $\bar x$, $n$ observations):
>
> posterior mean $=\dfrac{\kappa_0\,\mu_0 + n\,\bar x}{\kappa_0+n}$ (a weighted average: the prior counts as $\kappa_0$ observations).
>
> **Task.** For the **10 plants** of `ci_small`, compute the posterior mean for (i) a **weak** prior ($\mu_0=0.5$, $\kappa_0=0.01$) and (ii) a **wrong** prior ($\mu_0=0.3$, $\kappa_0=5$). The width of a 90% interval is roughly $1.8\times s/\sqrt{n+\kappa_0}$ ($s$ = standard deviation of the 10 values).
> **Question Q7b.** How much does the wrong prior move the answer, and why does this matter when data are scarce?

In [19]:
# The 10 plants of ci_small: one utilisation value per plant
pp = bf[bf["Plant ID"].isin(chosen)].groupby("Plant ID")[["production_ttpa", "capacity"]].sum()
x = (pp["production_ttpa"] / pp["capacity"]).to_numpy()
n, xbar, s = len(x), x.mean(), x.std(ddof=1)

for name, mu0, k0 in [("weak", 0.5, 0.01), ("wrong", 0.3, 5)]:
    post_mean = (k0 * mu0 + n * xbar) / (k0 + n)
    width = 1.8 * s / np.sqrt(n + k0)          # approximate 90% interval width
    print(name, post_mean, width, post_mean - width / 2, post_mean + width / 2)

weak 0.8208551411530377 0.08957232098614118 0.7760689806599671 0.8656413016461083
wrong 0.6474506641961271 0.07317205244049192 0.6108646379758811 0.684036690416373


**Your answer (Q7b (optional)):**

With a weak prior (μ0 = 0.5, κ0 = 0.01) the posterior mean is 82.1%, with a 90% interval of about [77.6%, 86.6%] (width about 9.0 points): the prior is worth only 0.01 observation, so the answer is essentially the data mean. With the wrong prior (μ0 = 0.3, κ0 = 5) it falls to 64.7%, with an interval of about [61.1%, 68.4%] (width about 7.3 points). The wrong prior therefore pulls the answer down by about 17 points, and its interval does not even contain the data mean. It also makes the interval narrower (7.3 vs 9.0 points), so we become confidently wrong. This matters when data are scarce: with only 10 plants, a prior worth 5 observations carries a third of the weight and can dominate the result while hiding the uncertainty. With more data, its weight would fade.

## B4. Simulation: a range for emissions

**Idea (Monte Carlo).** Instead of one value per input, use a *plausible range*, then **repeat the whole calculation many times**, each time with different plausible values. The cloud of results shows the uncertainty of the output.

Two uncertain inputs:

| Input | Plausible range | How to draw |
|---|---|---|
| **Utilisation** per technology | the bootstrap values of B3 | one bootstrap value per draw and technology (all technologies pooled if a technology has fewer than 5 plants) |
| **Emission factor** per technology | multiplier on your Part A factor, **assumed illustrative values** below | triangular distribution (low, mode = 1, high): `rng.triangular(low, 1, high, size=n)` |

| Technology | low | mode | high |
|---|---|---|---|
| BF-BOF | 0.85 | 1 | 1.20 |
| EAF | 0.65 | 1 | 2.00 |
| Other | 0.65 | 1 | 1.40 |

**Goal.** Create `ef_mult_ranges` (dict) and a function `simulate(n=1000, year=2030, vary_util=True, vary_ef=True, bf_cap_mult=1.0, util_mult=1.0, seed=0)` returning a dict with
* `national`: array of `n` national emissions (**MtCO2**) for `year`;
* `national_prod`: array of `n` national productions (**Mt**);
* `by_company`: DataFrame (`n` rows, one column per company, MtCO2).

> **Hint.**
> 1. Take the rows of `china_operational` for `year`. For every plant you need: capacity (missing = 0), technology, your Part A emission factor, company.
> 2. Draw a matrix `u` (n x 3) of utilisations and a matrix `mult` (n x 3) of multipliers (columns = BF-BOF, EAF, Other). If `vary_util=False`, use the pooled matched utilisation of each technology instead (sum of production / sum of capacity over all years); if `vary_ef=False`, use multipliers equal to 1.
> 3. Choose each plant's column by its technology: `u[:, tech_index_of_plant]` gives an (n x plants) matrix. Then production = capacity x utilisation (x `util_mult`), emissions = production x 1,000 x factor x multiplier / 1e6.
> 4. `bf_cap_mult` multiplies the capacity of BF-BOF plants (used for scenarios later).
> 5. Company totals: `pd.DataFrame(em_matrix, columns=company_of_each_plant).T.groupby(level=0).sum().T`.

In [20]:
techs = ["BF-BOF", "EAF", "Other"]
ef_mult_ranges = {"BF-BOF": (0.85, 1, 1.20), "EAF": (0.65, 1, 2.00), "Other": (0.65, 1, 1.40)}

# Bootstrap utilisation values and pooled utilisation per technology
# (all technologies pooled if a technology has fewer than 5 matched plants)
boot_vals, pooled_u = {}, {}
for t in techs:
    d = obs_matched[obs_matched["Technology_type"] == t]
    if d["Plant ID"].nunique() < 5:
        d = obs_matched
    boot_vals[t] = boot_mean_u(d, 1000)
    pooled_u[t] = d["production_ttpa"].sum() / d["capacity"].sum()

def simulate(n=1000, year=2030, vary_util=True, vary_ef=True, bf_cap_mult=1.0, util_mult=1.0, seed=0):
    rng = np.random.default_rng(seed)
    d = china_operational[china_operational["Year"] == year]
    cap = d["capacity"].fillna(0).to_numpy(dtype=float)
    tech = d["Technology_type"].map({t: i for i, t in enumerate(techs)}).to_numpy()
    cap = cap * np.where(tech == 0, bf_cap_mult, 1.0)          # scenario: retire BF-BOF capacity
    ef = d["Emission_factor_tCO2_per_t"].to_numpy(dtype=float)

    # Utilisation matrix (n x 3): one bootstrap value per draw and technology
    if vary_util:
        u = np.column_stack([rng.choice(boot_vals[t], size=n) for t in techs])
    else:
        u = np.tile([pooled_u[t] for t in techs], (n, 1))

    # Emission-factor multipliers (n x 3): triangular distributions
    if vary_ef:
        mult = np.column_stack([rng.triangular(*ef_mult_ranges[t], size=n) for t in techs])
    else:
        mult = np.ones((n, 3))

    prod = cap[None, :] * u[:, tech] * util_mult                 # ttpa, (n x plants)
    em = prod * 1000 * ef[None, :] * mult[:, tech] / 1e6         # MtCO2
    by_company = pd.DataFrame(em, columns=d["Company"].to_numpy()).T.groupby(level=0).sum().T
    return {"national": em.sum(1), "national_prod": prod.sum(1) / 1000, "by_company": by_company}

### Q8 [QUANT + QUAL] Read the result

(a) For **2030**, report the **median** and the **90% interval** (5th to 95th percentile) of national emissions, and the **relative half-width** = (upper - lower) / (2 x median). (b) Compare the median with your **Part A value for 2030** (sum of `company_emissions` for 2030, in Mt). Which one is closer to the simulation, and why do they differ so much (hint: B1)?

In [21]:
res = simulate(1000, year=2030)
lo, med, hi = np.percentile(res["national"], [5, 50, 95])
print(med, lo, hi, (hi - lo) / (2 * med))

part_a_2030 = company_emissions[company_emissions["Year"] == 2030]["estimated_emissions_tCO2"].sum() / 1e6
print(part_a_2030)

1150.6063944121106 1016.1158319125232 1314.7221836163412 0.12976042596060297
149.9853125751576


**Your answer (Q8):**

(a) For 2030, the median national emissions are about 1,151 MtCO2, with a 90% interval of about [1,016 ; 1,315] MtCO2. The relative half-width is (1,315 − 1,016) / (2 × 1,151) ≈ 13.0%.

(b) The Part A value for 2030 is only about 150 MtCO2, roughly 7.7 times lower, so the simulation is the more credible. They differ because Part A divides the production of the few plants that report (about 10% coverage, see B1) by the capacity of all plants, which gives a utilisation of only about 11% for BF-BOF. The simulation uses the matched utilisation (about 80%), so production and emissions are of the right order of magnitude.

### Q9 [QUANT + QUAL] Which input matters most?

Run `simulate` twice: once varying **only the utilisation** (`vary_ef=False`) and once varying **only the emission factor** (`vary_util=False`). For each, give the relative half-width for 2030. Which input creates more of the uncertainty? Could **more data** reduce it?

In [22]:
for name, kw in [("utilisation only", dict(vary_ef=False)),
                 ("emission factor only", dict(vary_util=False))]:
    x = simulate(1000, year=2030, **kw)["national"]
    lo, med, hi = np.percentile(x, [5, 50, 95])
    print(name, med, lo, hi, (hi - lo) / (2 * med))

utilisation only 1135.9416873249593 1077.3820869744343 1194.7870210282654 0.05167735957041473
emission factor only 1162.262921963629 1037.5471884717326 1295.758035136183 0.1110810823372935


**Your answer (Q9):**

Varying only the utilisation gives a relative half-width of about 5.2% (median 1,136 MtCO2, 90% interval [1,077 ; 1,195]), and varying only the emission factor gives about 11.1% (median 1,162 MtCO2, 90% interval [1,038 ; 1,296]). The emission factor therefore creates most of the uncertainty. The utilisation is estimated from 44 matched BF-BOF plants, so its bootstrap range is narrow, whereas the emission-factor multipliers are wide and apply to all plants of a technology at once, so the errors do not average out.

More data could reduce it: both are mainly epistemic uncertainty. Plant-level measured emission factors would shrink the biggest source, and more reporting plants would narrow the utilisation range. The part of utilisation that comes from year-to-year randomness (aleatoric) would remain.

### Q10 [QUANT + QUAL] Companies: can you say who is number one?

Using `by_company` of your 2030 run: (a) for the **top 3 companies by median**, give the median and the 90% interval; (b) compute the probability that the #1 company emits **more than** the #2 (fraction of simulations); (c) what would you write in a report instead of "Company X is the largest emitter"?

In [23]:
by_company = simulate(1000, year=2030)["by_company"]

# (a) top 3 by median, with 90% interval
top3 = by_company.median().sort_values(ascending=False).index[:3]
for c in top3:
    lo, m, hi = np.percentile(by_company[c], [5, 50, 95])
    print(c, m, lo, hi)

# (b) probability that #1 emits more than #2
print((by_company[top3[0]] > by_company[top3[1]]).mean())

Beijing Shougang Co Ltd [100.0%] 37.36784963973629 32.95543845039674 42.92436965938157
Maanshan Iron & Steel Co Ltd [100.0%] 29.236208731407835 25.783985074332524 33.583570987486056
Jingye Group Co Ltd [100.0%] 28.228020841238404 24.894844428530345 32.42546769545917
1.0


**Your answer (Q10):**

(a) Top 3 by median in 2030: Beijing Shougang 37.4 MtCO2 (90% interval 33.0-42.9), Maanshan Iron & Steel 29.2 (25.8-33.6) and Jingye Group 28.2 (24.9-32.4). The intervals of #2 and #3 overlap almost completely.

(b) The probability that #1 emits more than #2 is 1.00 (all simulations). This is an artefact of the model, not real certainty: in each draw, the same utilisation and the same emission-factor multiplier apply to all companies of a technology, so they move together and only their capacity decides the ranking. The model has no plant-level randomness and ignores data errors.

(c) Instead of “Company X is the largest emitter”, I would write: “Within this model, Beijing Shougang is the largest emitter, at about 37 MtCO2 (90% interval 33-43). Ranks 2 and 3 are close (28-29 MtCO2) and their intervals overlap, and the ranking is conditional on the model’s assumptions and on the quality of the ownership data.”

## B5. Is the interval honest? Test it

A 90% interval should contain the true value in about 90% of cases. We have one case where the truth is known: **official 2023 production**.

### Q11 [QUANT + QUAL] Does the official value fall inside?

Run `simulate(1000, year=2023)` and look at `national_prod` (Mt). (a) Give its 90% interval and say whether the official 2023 value (1,020 Mt) falls inside. (b) If not, what does this tell you about the interval, **even though it was computed correctly**? (c) What would you do about it?

In [24]:
res23 = simulate(1000, year=2023)
lo, med, hi = np.percentile(res23["national_prod"], [5, 50, 95])
print(lo, med, hi)
print("official inside:", lo <= 1019 <= hi)

607.7059570308347 636.5838615590455 667.7855531129005
official inside: False


**Your answer (Q11):**

(a) The 90% interval of simulated 2023 production is about [608 ; 668] Mt (median about 637 Mt). The official value (1,019 Mt) does not fall inside: it is about 350 Mt above the upper bound.

(b) The interval was computed correctly, but it is not honest: it only reflects the uncertainty of the utilisation (bootstrap) and of the emission factors, given the model’s structure. It ignores the larger error, the missing capacity: the dataset covers only 775 Mt of capacity (implied utilisation of 131%, see B1), so even a perfect utilisation could not reach the official production. A 90% interval is only valid if the assumptions are right, and here one of them (complete capacity coverage) is wrong. This is a model-structure (epistemic) error, not sampling noise.

(c) I would (1) add the missing capacity or a coverage factor calibrated on official statistics (for example by scaling the model so that it reproduces the official national total), (2) include the uncertainty of this coverage factor in the simulation so the interval widens, and (3) backtest the interval on every year where an official value exists, not only 2023.

## B6. Scenarios

A **scenario** is a "what if" story. We do **not** attach a probability to it; we run each one through the same simulation and compare.

| Scenario | Story | How |
|---|---|---|
| **S0** | no change | as is |
| **S1** | **30% of BF-BOF capacity is retired** (by 2027, still true in 2030) | `bf_cap_mult=0.7` |
| **S2** | **steel demand drops**: utilisation 10% lower | `util_mult=0.9` |

### Q12 [QUANT + QUAL] Scenario gap vs interval width

(a) Run the three scenarios for 2030 and give the **median and 90% interval** of each. (b) Compare the **gap between** scenario medians with the **width of the interval** within a scenario. Which is bigger? (c) Why would it be misleading to average the three scenarios into one probability-weighted number?

In [25]:
scenarios = {"S0": {}, "S1": dict(bf_cap_mult=0.7), "S2": dict(util_mult=0.9)}
draws = {k: simulate(1000, year=2030, **kw)["national"] for k, kw in scenarios.items()}

for k, x in draws.items():
    lo, m, hi = np.percentile(x, [5, 50, 95])
    print(k, m, lo, hi, hi - lo)

S0 1150.6063944121106 1016.1158319125232 1314.7221836163412 298.60635170381806
S1 818.0607445119897 724.0104604120083 933.5246644473783 209.51420403536997
S2 1035.5457549708992 914.5042487212714 1183.2499652547071 268.7457165334357


**Your answer (Q12):**

(a) S0: median about 1,151 MtCO2, 90% interval [1,016 ; 1,315]. S1: median about 818, interval [724 ; 934]. S2: median about 1,036, interval [915 ; 1,183].

(b) The gap between S0 and S1 medians (about 333 Mt) is bigger than the width of the interval within a scenario (210-299 Mt), so the retirement scenario clearly changes the result. The gap between S0 and S2 (about 115 Mt) is smaller than the within-scenario width, so their intervals overlap and the demand scenario is hard to separate from the statistical noise.

(c) It would be misleading to average the scenarios into one probability-weighted number because scenarios are “what if” stories about future choices, not random events: we have no basis to give them probabilities. The weights would be arbitrary, and the result would hide the spread of outcomes depending on the choice, which is what a decision-maker needs, and would mix scenario uncertainty with statistical uncertainty.

## B7. Communicate

Words used by the IPCC for probabilities: **virtually certain** 99-100%, **very likely** 90-100%, **likely** 66-100%, **about as likely as not** 33-66%, **unlikely** 10-33%, **very unlikely** 1-10%, **exceptionally unlikely** 0-1%.

### Q13 [QUANT + QUAL] A probability, a word, and a note

(a) For each scenario, compute the probability that **2030 emissions exceed the 2023 median** (from your `simulate(..., year=2023)`) and give the matching word. (b) Write a note of **about 80 words** for a bank's risk committee about China's steel emissions in 2030. Include: the **median and 90% interval** (baseline), **one sentence on the scenarios**, the **biggest source of uncertainty**, **one limitation** of the model and **one recommended action**. Plain language, no jargon.

> **Hint.** `(draws > threshold).mean()`.

In [26]:
threshold = np.median(simulate(1000, year=2023)["national"])   # 2023 median emissions (MtCO2)

for k, x in draws.items():
    print(k, (x > threshold).mean())

S0 0.426
S1 0.0
S2 0.071


**Your answer (Q13):**

(a) Probability that 2030 emissions exceed the 2023 median: S0 ≈ 43% → “about as likely as not” (33-66%); S1 = 0% → “exceptionally unlikely” (0-1%); S2 ≈ 7% → “very unlikely” (1-10%).

(b) Note for the risk committee:
“Baseline estimate for China’s steel emissions in 2030: about 1,150 MtCO2, with a 90% range of 1,016 to 1,315. Retiring 30% of blast-furnace capacity would cut this to about 820, while 10% lower demand gives about 1,040. The biggest source of uncertainty is the emission factors of the plants. A key limitation: the data miss part of China’s capacity, so our 2023 production is far below the official figure. We recommend calibrating the model to official statistics and collecting measured plant-level emission factors.”